# 🌐 Machine Translation using Encoder–Decoder Network
## NLP Assignment 2 — BITS Pilani M.Tech (AI/ML)

**Languages Chosen:** English → Hindi (`en` → `hi`)

**Dataset:** [Samanantar](https://www.kaggle.com/datasets/mathurinache/samanantar) — The largest publicly available parallel corpora collection for 11 Indic languages

**Architecture:** GRU-based Encoder–Decoder with Teacher Forcing

---

## 👥 Team Members

| Sl. No | Name | BITS ID |
|--------|------|----------|
| 1 | Jayakrishnan J | 2025AA05072 |
| 2 | Nilesh Lohar | 2025AA05156 |
| 3 | Abhinav Kumar | 2025AA05817 |
| 4 | Amey Mudras | 2025AA0552 |
| 5 | Neha Raj | 2025AA05672 |

## 🔧 Environment Setup

Run on **BITS OSHA Cloud Lab** (preferred) or **Google Colab** with GPU.

**Kaggle dataset setup:**
1. Go to [kaggle.com](https://www.kaggle.com) → Account → API → *Create New Token* → download `kaggle.json`
2. Upload `kaggle.json` to `~/.kaggle/` (Colab: run `from google.colab import files; files.upload()`)
3. The dataset cell will auto-download.

In [ ]:
# Install required packages
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

install('sacrebleu')
install('kaggle')
print('✓ Dependencies installed')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
import random
import os
import time
import unicodedata
import zipfile
from collections import Counter
from typing import List, Tuple, Dict, Optional
import sacrebleu as sb
from sacrebleu.metrics import BLEU
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ── Device ───────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## Task 1: Dataset Preparation (2 Marks)

### Dataset: Samanantar — English ↔ Hindi

Samanantar is a parallel corpus of 8M+ English–Hindi sentence pairs.  
We use a **50,000-pair subset** for tractable training on cloud lab GPUs.

| Property | Value |
|----------|-------|
| Source language | English (en) |
| Target language | Hindi (hi) |
| Training pairs | 45,000 |
| Test pairs | 5,000 |
| Max sequence length | 50 tokens |

In [ ]:
DATA_DIR = './data'
EN_FILE  = os.path.join(DATA_DIR, 'en-hi', 'train.en')
HI_FILE  = os.path.join(DATA_DIR, 'en-hi', 'train.hi')

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(EN_FILE):
    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_json):
        print('Downloading Samanantar dataset …')
        os.system(f'kaggle datasets download -d mathurinache/samanantar -p {DATA_DIR} -q')
        zip_path = os.path.join(DATA_DIR, 'samanantar.zip')
        if os.path.exists(zip_path):
            print('Extracting …')
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA_DIR)
            print('✓ Extracted to', DATA_DIR)
        else:
            print('⚠ ZIP not found — check Kaggle credentials / quota')
    else:
        print('⚠ kaggle.json not found at ~/.kaggle/kaggle.json')
        print('  Upload it and re-run, or place en-hi/train.en + train.hi in ./data/')
else:
    print('✓ Dataset files already present')

In [ ]:
# ── Built-in fallback (200 real en-hi pairs) ─────────────────────
# Used automatically when the Kaggle dataset is not available
FALLBACK_PAIRS = [
    ('the government announced new policies today', 'सरकार ने आज नई नीतियों की घोषणा की'),
    ('she is studying at the university', 'वह विश्वविद्यालय में पढ़ रही है'),
    ('the children are playing in the park', 'बच्चे पार्क में खेल रहे हैं'),
    ('he went to the market to buy vegetables', 'वह सब्जियाँ खरीदने बाजार गया'),
    ('the train arrives at ten in the morning', 'ट्रेन सुबह दस बजे आती है'),
    ('india is a large and diverse country', 'भारत एक बड़ा और विविध देश है'),
    ('the doctor advised him to rest for a week', 'डॉक्टर ने उसे एक सप्ताह आराम करने की सलाह दी'),
    ('please close the door when you leave', 'जाते समय कृपया दरवाजा बंद करें'),
    ('the sun sets in the west', 'सूर्य पश्चिम में अस्त होता है'),
    ('water is essential for life', 'जल जीवन के लिए आवश्यक है'),
    ('the farmer works hard in the fields', 'किसान खेतों में कड़ी मेहनत करता है'),
    ('she reads a book every night before sleeping', 'वह सोने से पहले हर रात एक किताब पढ़ती है'),
    ('the students passed their examinations', 'छात्रों ने अपनी परीक्षाएँ उत्तीर्ण कीं'),
    ('it is raining heavily outside', 'बाहर भारी बारिश हो रही है'),
    ('the river flows through the valley', 'नदी घाटी से होकर बहती है'),
    ('he speaks three languages fluently', 'वह तीन भाषाएँ धाराप्रवाह बोलता है'),
    ('the hospital is very far from here', 'अस्पताल यहाँ से बहुत दूर है'),
    ('they celebrated the festival with joy', 'उन्होंने त्योहार खुशी के साथ मनाया'),
    ('the old man sat under the tree', 'बूढ़ा आदमी पेड़ के नीचे बैठा था'),
    ('the teacher explained the lesson clearly', 'शिक्षक ने पाठ को स्पष्ट रूप से समझाया'),
    ('i will visit my parents next week', 'मैं अगले सप्ताह अपने माता-पिता से मिलने जाऊँगा'),
    ('the birds fly south in winter', 'पक्षी सर्दियों में दक्षिण की ओर उड़ते हैं'),
    ('she cooked a delicious meal for the family', 'उसने परिवार के लिए एक स्वादिष्ट भोजन पकाया'),
    ('the library has thousands of books', 'पुस्तकालय में हजारों किताबें हैं'),
    ('he finished his work before the deadline', 'उसने समय सीमा से पहले अपना काम पूरा किया'),
    ('the match was very exciting to watch', 'मैच देखना बहुत रोमांचक था'),
    ('she bought a new dress for the wedding', 'उसने शादी के लिए एक नई पोशाक खरीदी'),
    ('the factory produces thousands of cars daily', 'कारखाना प्रतिदिन हजारों कारें बनाता है'),
    ('please speak slowly so i can understand', 'कृपया धीरे बोलें ताकि मैं समझ सकूँ'),
    ('the mountain peak was covered in snow', 'पर्वत की चोटी बर्फ से ढकी थी'),
    ('he bought fresh fruits from the shop', 'उसने दुकान से ताजे फल खरीदे'),
    ('the weather is pleasant today', 'आज मौसम सुहावना है'),
    ('the election results were announced yesterday', 'कल चुनाव परिणाम घोषित किए गए'),
    ('she learned to drive last year', 'उसने पिछले साल गाड़ी चलाना सीखा'),
    ('the movie was released last friday', 'फिल्म पिछले शुक्रवार को रिलीज हुई'),
    ('the price of petrol has increased', 'पेट्रोल की कीमत बढ़ गई है'),
    ('the garden is full of beautiful flowers', 'बगीचा सुंदर फूलों से भरा है'),
    ('he donated money to the charity', 'उसने दान में पैसे दिए'),
    ('they built a new bridge over the river', 'उन्होंने नदी पर एक नया पुल बनाया'),
    ('the baby was sleeping peacefully', 'बच्चा शांति से सो रहा था'),
    ('she won the first prize in the competition', 'उसने प्रतियोगिता में पहला पुरस्कार जीता'),
    ('the computer needs to be repaired', 'कंप्यूटर को ठीक करने की जरूरत है'),
    ('they went for a walk in the evening', 'वे शाम को टहलने गए'),
    ('the school is closed on sundays', 'रविवार को स्कूल बंद रहता है'),
    ('he is learning to play the guitar', 'वह गिटार बजाना सीख रहा है'),
    ('the news was broadcast on television', 'समाचार टेलीविजन पर प्रसारित किया गया'),
    ('she has been working here for five years', 'वह पाँच वर्षों से यहाँ काम कर रही है'),
    ('the conference will begin tomorrow morning', 'सम्मेलन कल सुबह शुरू होगा'),
    ('he forgot his umbrella at home', 'वह अपनी छाता घर भूल गया'),
    ('the cow gives milk every day', 'गाय प्रतिदिन दूध देती है'),
    ('they are planning to open a new restaurant', 'वे एक नया रेस्तरां खोलने की योजना बना रहे हैं'),
    ('the road is under construction', 'सड़क निर्माणाधीन है'),
    ('she called her mother from the airport', 'उसने हवाई अड्डे से अपनी माँ को फोन किया'),
    ('the mango tree grows in warm climates', 'आम का पेड़ गर्म जलवायु में उगता है'),
    ('he reads the newspaper every morning', 'वह हर सुबह अखबार पढ़ता है'),
    ('the children enjoyed the picnic very much', 'बच्चों ने पिकनिक का बहुत आनंद लिया'),
    ('she takes medicine twice a day', 'वह दिन में दो बार दवाई लेती है'),
    ('the bank is open from nine to five', 'बैंक नौ बजे से पाँच बजे तक खुला रहता है'),
    ('he planted a tree in his garden', 'उसने अपने बगीचे में एक पेड़ लगाया'),
    ('the cat sat on the mat', 'बिल्ली चटाई पर बैठी'),
    ('she sent a letter to her friend', 'उसने अपने दोस्त को एक पत्र भेजा'),
    ('the temple was visited by thousands of pilgrims', 'मंदिर में हजारों तीर्थयात्री आए'),
    ('he repaired the broken chair', 'उसने टूटी हुई कुर्सी ठीक की'),
    ('the bus was late by thirty minutes', 'बस तीस मिनट देर से आई'),
    ('she prepared for the interview thoroughly', 'उसने साक्षात्कार की अच्छी तरह तैयारी की'),
    ('the dog ran after the ball', 'कुत्ता गेंद के पीछे दौड़ा'),
    ('he ate his dinner quickly', 'उसने जल्दी से अपना रात का खाना खाया'),
    ('the festival brings people together', 'त्योहार लोगों को एक साथ लाता है'),
    ('she passed her driving test on the first attempt', 'उसने पहली बार में ड्राइविंग टेस्ट पास किया'),
    ('the prime minister addressed the nation', 'प्रधानमंत्री ने राष्ट्र को संबोधित किया'),
    ('the student submitted the assignment on time', 'छात्र ने असाइनमेंट समय पर जमा किया'),
    ('she wore a beautiful saree to the party', 'उसने पार्टी में एक सुंदर साड़ी पहनी'),
    ('the lion is the king of the jungle', 'शेर जंगल का राजा है'),
    ('he completed his graduation this year', 'उसने इस वर्ष अपना स्नातक पूरा किया'),
    ('the office is closed on public holidays', 'सार्वजनिक अवकाश पर कार्यालय बंद रहता है'),
    ('she goes jogging every morning in the park', 'वह हर सुबह पार्क में जॉगिंग करती है'),
    ('the flood damaged many houses in the village', 'बाढ़ ने गाँव के कई घरों को नुकसान पहुँचाया'),
    ('he is writing a novel about his experiences', 'वह अपने अनुभवों के बारे में एक उपन्यास लिख रहा है'),
    ('the market opens early in the morning', 'बाजार सुबह जल्दी खुलता है'),
    ('she made tea for all the guests', 'उसने सभी मेहमानों के लिए चाय बनाई'),
    ('the election was held last month', 'पिछले महीने चुनाव हुए'),
    ('he saved money to buy a new phone', 'उसने नया फोन खरीदने के लिए पैसे बचाए'),
    ('the children went on a field trip today', 'बच्चे आज भ्रमण पर गए'),
    ('she helps her mother with household chores', 'वह घर के कामों में अपनी माँ की मदद करती है'),
    ('the river was flooded after heavy rain', 'भारी बारिश के बाद नदी में बाढ़ आ गई'),
    ('he turned off the lights before leaving', 'जाने से पहले उसने लाइटें बंद कर दीं'),
    ('the shop sells fresh vegetables and fruits', 'दुकान ताजी सब्जियाँ और फल बेचती है'),
    ('she sang a beautiful song at the concert', 'उसने कार्यक्रम में एक सुंदर गाना गाया'),
    ('the mechanic fixed the car in two hours', 'मैकेनिक ने दो घंटे में कार ठीक की'),
    ('he woke up late and missed the bus', 'वह देर से उठा और बस छूट गई'),
    ('the government built new hospitals in the district', 'सरकार ने जिले में नए अस्पताल बनाए'),
    ('she is working on her research project', 'वह अपने शोध प्रोजेक्ट पर काम कर रही है'),
    ('the news spread quickly through the city', 'शहर में खबर तेजी से फैल गई'),
    ('he climbed the stairs to the third floor', 'वह सीढ़ियाँ चढ़कर तीसरी मंजिल पर गया'),
    ('the soldier was awarded a medal for bravery', 'सैनिक को बहादुरी के लिए पदक दिया गया'),
    ('she cooked rice and lentils for lunch', 'उसने दोपहर के खाने में चावल और दाल बनाई'),
    ('the teacher gave extra homework to the students', 'शिक्षक ने छात्रों को अतिरिक्त गृहकार्य दिया'),
    ('he is preparing for his job interview', 'वह अपने नौकरी के साक्षात्कार की तैयारी कर रहा है'),
    ('the wedding ceremony was held in the temple', 'विवाह समारोह मंदिर में हुआ'),
    ('she borrowed a book from the library', 'उसने पुस्तकालय से एक किताब उधार ली'),
    ('the stadium was full of cheering fans', 'स्टेडियम उत्साहित प्रशंसकों से भरा था'),
    ('he received an award for his outstanding work', 'उसे अपने उत्कृष्ट कार्य के लिए पुरस्कार मिला'),
    ('the pilot landed the plane safely', 'पायलट ने विमान को सुरक्षित रूप से उतारा'),
    ('she decorated the house for the festival', 'उसने त्योहार के लिए घर को सजाया'),
    ('the scientist made an important discovery', 'वैज्ञानिक ने एक महत्वपूर्ण खोज की'),
    ('he takes the metro to work every day', 'वह हर दिन काम पर मेट्रो से जाता है'),
    ('the garden needs to be watered daily', 'बगीचे को रोज पानी देने की जरूरत है'),
    ('she sent flowers to her sister on her birthday', 'उसने अपनी बहन को उसके जन्मदिन पर फूल भेजे'),
    ('the police arrested the thief last night', 'पुलिस ने कल रात चोर को गिरफ्तार किया'),
    ('he studied hard and got good marks', 'उसने कड़ी मेहनत की और अच्छे अंक प्राप्त किए'),
    ('the village is surrounded by forests', 'गाँव जंगलों से घिरा हुआ है'),
    ('she finished sewing the dress in one day', 'उसने एक दिन में पोशाक सिलकर तैयार कर दी'),
    ('the company launched a new product today', 'कंपनी ने आज एक नया उत्पाद लॉन्च किया'),
    ('he washes his hands before every meal', 'वह हर भोजन से पहले हाथ धोता है'),
    ('the airport is busy during the holiday season', 'छुट्टी के मौसम में हवाई अड्डा व्यस्त रहता है'),
    ('she smiled when she heard the good news', 'अच्छी खबर सुनकर वह मुस्कुराई'),
    ('the old building was demolished last week', 'पुरानी इमारत पिछले सप्ताह ध्वस्त की गई'),
    ('he played football with his friends', 'उसने अपने दोस्तों के साथ फुटबॉल खेली'),
    ('the river provides water to many villages', 'नदी कई गाँवों को पानी प्रदान करती है'),
    ('she packed her bags for the trip', 'उसने यात्रा के लिए अपना सामान पैक किया'),
    ('the electricity was cut off for three hours', 'तीन घंटे के लिए बिजली कट गई'),
    ('he studied medicine at a prestigious college', 'उसने एक प्रतिष्ठित कॉलेज में चिकित्सा का अध्ययन किया'),
    ('the cat caught a mouse in the kitchen', 'बिल्ली ने रसोई में एक चूहा पकड़ा'),
    ('she runs a small business from home', 'वह घर से एक छोटा व्यवसाय चलाती है'),
    ('the street lights were turned on at dusk', 'शाम को सड़क की रोशनी चालू की गई'),
    ('he invited his colleagues to his birthday party', 'उसने अपने सहकर्मियों को अपनी जन्मदिन पार्टी में आमंत्रित किया'),
    ('the plants need sunlight to grow', 'पौधों को बढ़ने के लिए धूप चाहिए'),
    ('she applied for a scholarship to study abroad', 'उसने विदेश में पढ़ने के लिए छात्रवृत्ति के लिए आवेदन किया'),
    ('the museum exhibits ancient artifacts', 'संग्रहालय प्राचीन कलाकृतियाँ प्रदर्शित करता है'),
    ('he drank two glasses of water after the run', 'दौड़ के बाद उसने दो गिलास पानी पिया'),
    ('the crowd cheered for the winning team', 'भीड़ ने जीतने वाली टीम के लिए चीयर किया'),
    ('she organized a surprise party for her husband', 'उसने अपने पति के लिए एक सरप्राइज पार्टी का आयोजन किया'),
    ('the bridge collapsed due to heavy floods', 'भारी बाढ़ के कारण पुल ढह गया'),
    ('he fixed the leaking pipe in the bathroom', 'उसने बाथरूम में टपकती पाइप ठीक की'),
    ('the teacher praised the student for his efforts', 'शिक्षक ने उसकी मेहनत के लिए छात्र की प्रशंसा की'),
    ('she volunteered at the local school', 'उसने स्थानीय स्कूल में स्वयंसेवा की'),
    ('the forest fire spread rapidly in dry weather', 'सूखे मौसम में जंगल की आग तेजी से फैली'),
    ('he met his old friend at the station', 'उसने स्टेशन पर अपने पुराने दोस्त से मुलाकात की'),
    ('the farmer harvested the crop on time', 'किसान ने समय पर फसल काटी'),
    ('she designed a beautiful website for the company', 'उसने कंपनी के लिए एक सुंदर वेबसाइट डिज़ाइन की'),
    ('the baby started walking at ten months', 'बच्चे ने दस महीने में चलना शुरू किया'),
    ('he donated blood at the blood donation camp', 'उसने रक्तदान शिविर में रक्तदान किया'),
    ('the rocket was launched successfully into orbit', 'रॉकेट को सफलतापूर्वक कक्षा में लॉन्च किया गया'),
    ('she made a presentation in the board meeting', 'उसने बोर्ड बैठक में एक प्रस्तुति दी'),
    ('the new metro line connects the airport to the city', 'नई मेट्रो लाइन हवाई अड्डे को शहर से जोड़ती है'),
    ('he cooked breakfast for the entire family', 'उसने पूरे परिवार के लिए नाश्ता बनाया'),
    ('the solar panels were installed on the rooftop', 'छत पर सोलर पैनल लगाए गए'),
    ('she wrote a poem for her grandmother', 'उसने अपनी नानी के लिए एक कविता लिखी'),
    ('the match was tied after extra time', 'अतिरिक्त समय के बाद मैच बराबरी पर छूटा'),
    ('he worked late into the night to finish the report', 'उसने रिपोर्ट खत्म करने के लिए देर रात तक काम किया'),
    ('the election commission announced the voting schedule', 'चुनाव आयोग ने मतदान कार्यक्रम की घोषणा की'),
    ('she bought a laptop for her college studies', 'उसने कॉलेज की पढ़ाई के लिए एक लैपटॉप खरीदा'),
    ('the poet recited verses at the cultural event', 'कवि ने सांस्कृतिक कार्यक्रम में कविताएँ सुनाईं'),
    ('he apologized for his mistake sincerely', 'उसने अपनी गलती के लिए सच्चे दिल से माफी माँगी'),
    ('the ambulance arrived within ten minutes', 'एंबुलेंस दस मिनट के भीतर आ गई'),
    ('she plays the piano very well', 'वह पियानो बहुत अच्छा बजाती है'),
    ('the night sky was full of stars', 'रात का आकाश तारों से भरा था'),
    ('he sent a gift to his niece on diwali', 'उसने दीवाली पर अपनी भतीजी को उपहार भेजा'),
    ('the children performed a play at the annual day', 'बच्चों ने वार्षिक दिवस पर एक नाटक प्रस्तुत किया'),
    ('she got promoted to a senior position', 'उसे एक वरिष्ठ पद पर पदोन्नत किया गया'),
    ('the team worked together to solve the problem', 'टीम ने समस्या को हल करने के लिए मिलकर काम किया'),
    ('he is studying for his entrance examination', 'वह अपनी प्रवेश परीक्षा के लिए पढ़ रहा है'),
    ('the price of onions has doubled this month', 'इस महीने प्याज की कीमत दोगुनी हो गई है'),
    ('she received a scholarship for higher education', 'उसे उच्च शिक्षा के लिए छात्रवृत्ति मिली'),
    ('the new law was passed by the parliament', 'संसद ने नया कानून पास किया'),
    ('he plays cricket every sunday with his friends', 'वह हर रविवार अपने दोस्तों के साथ क्रिकेट खेलता है'),
    ('the sun rose at six in the morning today', 'आज सुबह छह बजे सूर्योदय हुआ'),
    ('she helped the lost child find his parents', 'उसने खोए हुए बच्चे को उसके माता-पिता को ढूँढने में मदद की'),
    ('the patient recovered fully after surgery', 'सर्जरी के बाद मरीज पूरी तरह ठीक हो गया'),
    ('he painted the walls of his room blue', 'उसने अपने कमरे की दीवारें नीली रंग से रंगी'),
    ('the college held a cultural fest last weekend', 'कॉलेज ने पिछले सप्ताहांत एक सांस्कृतिक उत्सव आयोजित किया'),
    ('she takes her dog for a walk every evening', 'वह हर शाम अपने कुत्ते को टहलाने ले जाती है'),
    ('the government reduced taxes on electric vehicles', 'सरकार ने इलेक्ट्रिक वाहनों पर कर कम किया'),
    ('he applied for a loan to start his business', 'उसने अपना व्यवसाय शुरू करने के लिए ऋण के लिए आवेदन किया'),
    ('the peacock danced in the rain', 'मोर बारिश में नाचा'),
    ('she laughed at the funny joke', 'वह मजेदार चुटकुले पर हँसी'),
    ('the policeman helped the injured man', 'पुलिसकर्मी ने घायल आदमी की मदद की'),
    ('he learned a new programming language online', 'उसने ऑनलाइन एक नई प्रोग्रामिंग भाषा सीखी'),
    ('the news anchor read the headlines at nine', 'समाचार वाचक ने नौ बजे सुर्खियाँ पढ़ीं'),
    ('she cleaned the entire house in the morning', 'उसने सुबह पूरा घर साफ किया'),
    ('the river overflowed its banks during the monsoon', 'मानसून के दौरान नदी अपने किनारे से उफन गई'),
    ('he got lost in the new city', 'वह नए शहर में रास्ता भटक गया'),
    ('the lawyer presented strong evidence in court', 'वकील ने अदालत में मजबूत सबूत पेश किए'),
    ('she cut the vegetables for the soup', 'उसने सूप के लिए सब्जियाँ काटीं'),
    ('the students went on a picnic to the zoo', 'छात्र चिड़ियाघर में पिकनिक पर गए'),
    ('he donated old clothes to the poor', 'उसने गरीबों को पुराने कपड़े दान किए'),
    ('the hospital provides free treatment to the poor', 'अस्पताल गरीबों को मुफ्त इलाज देता है'),
    ('she read the terms and conditions carefully', 'उसने शर्तें और नियम ध्यान से पढ़े'),
    ('the forest is home to many wild animals', 'जंगल कई जंगली जानवरों का घर है'),
    ('he arrived at the station just in time', 'वह ठीक समय पर स्टेशन पहुँचा'),
    ('the minister inaugurated the new hospital wing', 'मंत्री ने नए अस्पताल विंग का उद्घाटन किया'),
    ('she learned yoga from a certified instructor', 'उसने एक प्रमाणित प्रशिक्षक से योग सीखा'),
    ('the car broke down on the highway', 'राजमार्ग पर कार खराब हो गई'),
    ('he renewed his passport at the passport office', 'उसने पासपोर्ट कार्यालय में अपना पासपोर्ट नवीनीकृत कराया'),
    ('the children exchanged gifts on christmas', 'बच्चों ने क्रिसमस पर उपहारों का आदान-प्रदान किया')
]

MAX_SAMPLES = 50000
MAX_LEN     = 30      # keep sequences short for demo

def load_parallel_corpus(en_path, hi_path, max_samples):
    with open(en_path, 'r', encoding='utf-8') as f:
        en_lines = [l.strip() for l in f.readlines()[:max_samples * 2] if l.strip()]
    with open(hi_path, 'r', encoding='utf-8') as f:
        hi_lines = [l.strip() for l in f.readlines()[:max_samples * 2] if l.strip()]
    pairs = [(e, h) for e, h in zip(en_lines, hi_lines) if e and h]
    return pairs[:max_samples]

try:
    pairs = load_parallel_corpus(EN_FILE, HI_FILE, MAX_SAMPLES)
    USING_REAL_DATA = True
    print(f'✓ Loaded {len(pairs):,} English-Hindi sentence pairs from Samanantar')
except FileNotFoundError:
    pairs = FALLBACK_PAIRS * 10   # repeat to get ~2000 pairs for training demo
    # Add slight variation to avoid 100% duplicates
    pairs = list(set(pairs))
    USING_REAL_DATA = False
    print(f'⚠ Dataset not found — using built-in sample data ({len(pairs)} pairs)')
    print('  Note: BLEU scores will be low; download Samanantar for real results')
    MAX_LEN = 20   # shorter for demo

# Preview
print(f'\nSample pairs:')
for en, hi in random.sample(pairs, min(5, len(pairs))):
    print(f'  EN: {en}')
    print(f'  HI: {hi}\n')

In [ ]:
# ── Preprocessing functions ──────────────────────────────────────

def preprocess_english(text: str) -> str:
    """
    English preprocessing:
    1. Convert to lowercase
    2. Normalize unicode (accents etc.)
    3. Remove non-alphanumeric symbols (keep spaces)
    4. Collapse multiple spaces
    """
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')  # strip diacritics
    text = re.sub(r"[^a-z0-9\s']", ' ', text)     # keep alphanumeric + apostrophe
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def preprocess_hindi(text: str) -> str:
    """
    Hindi (Devanagari) preprocessing:
    1. Keep Devanagari Unicode range U+0900–U+097F
    2. Keep Devanagari danda (।) and spaces
    3. Remove everything else
    4. Collapse multiple spaces
    Note: Hindi has no case — no lowercase step needed.
    """
    text = re.sub(r'[^\u0900-\u097F\s\u0964]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Test preprocessing
test_cases = [
    ("The Government Announced NEW policies!!! @2024", "सरकार ने नई नीतियों की घोषणा की! @2024"),
    ("He'll be there at 5pm... #Urgent",              "वह 5 बजे वहाँ होगा... #जरूरी"),
]
print('Preprocessing examples:')
print('─' * 60)
for en, hi in test_cases:
    print(f'Raw  EN: {en}')
    print(f'Proc EN: {preprocess_english(en)}')
    print(f'Raw  HI: {hi}')
    print(f'Proc HI: {preprocess_hindi(hi)}')
    print()

# ── Apply to all pairs ───────────────────────────────────────────
print('Applying preprocessing …')
processed_pairs = []
for en, hi in pairs:
    en_p = preprocess_english(en)
    hi_p = preprocess_hindi(hi)
    en_t = en_p.split()
    hi_t = hi_p.split()
    if 2 <= len(en_t) <= MAX_LEN and 2 <= len(hi_t) <= MAX_LEN:
        processed_pairs.append((en_p, hi_p))

print(f'✓ After filtering : {len(processed_pairs):,} pairs  '
      f'(removed {len(pairs)-len(processed_pairs):,} too-long/empty)')

# Length distribution
en_lens = [len(p[0].split()) for p in processed_pairs]
hi_lens = [len(p[1].split()) for p in processed_pairs]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, lens, lang, color in [
    (axes[0], en_lens, 'English', 'steelblue'),
    (axes[1], hi_lens, 'Hindi',   'coral')
]:
    ax.hist(lens, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(lens), color='black', linestyle='--',
               label=f'Mean={np.mean(lens):.1f}')
    ax.set_title(f'{lang} Sentence Length Distribution', fontsize=12)
    ax.set_xlabel('Tokens'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Sentence Length Distributions — Samanantar en-hi', fontsize=13)
plt.tight_layout()
plt.savefig('length_dist.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'English — mean: {np.mean(en_lens):.1f} | max: {max(en_lens)} | min: {min(en_lens)}')
print(f'Hindi   — mean: {np.mean(hi_lens):.1f} | max: {max(hi_lens)} | min: {min(hi_lens)}')

In [ ]:
# ── Special tokens ───────────────────────────────────────────────
PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


class Vocabulary:
    """
    Bidirectional word ↔ index mapping.

    Special tokens (always present):
      <PAD> = 0  padding — keeps embedding gradients zero
      <SOS> = 1  start-of-sequence — decoder's first input token
      <EOS> = 2  end-of-sequence — signals decoder to stop
      <UNK> = 3  unknown — replaces OOV words at inference
    """
    def __init__(self, name: str):
        self.name = name
        self.w2i: Dict[str, int] = {PAD: 0, SOS: 1, EOS: 2, UNK: 3}
        self.i2w: Dict[int, str] = {0: PAD, 1: SOS, 2: EOS, 3: UNK}
        self._freq: Counter = Counter()
        self.n_words: int = 4

    def count(self, sentence: str):
        for w in sentence.split():
            self._freq[w] += 1

    def build(self, min_freq: int = 2, max_vocab: Optional[int] = None):
        for w, c in self._freq.most_common(max_vocab):
            if c < min_freq:
                break
            if w not in self.w2i:
                self.w2i[w] = self.n_words
                self.i2w[self.n_words] = w
                self.n_words += 1

    def encode(self, sentence: str, add_sos=False, add_eos=True) -> List[int]:
        ids = [SOS_IDX] if add_sos else []
        ids += [self.w2i.get(w, UNK_IDX) for w in sentence.split()]
        if add_eos:
            ids.append(EOS_IDX)
        return ids

    def decode(self, indices: List[int]) -> str:
        words = []
        for i in indices:
            w = self.i2w.get(i, UNK)
            if w in {PAD, SOS}:
                continue
            if w == EOS:
                break
            words.append(w)
        return ' '.join(words)

    def __len__(self):
        return self.n_words

    def __repr__(self):
        return f"Vocabulary('{self.name}', size={self.n_words})"


# ── Train / test split ───────────────────────────────────────────
random.shuffle(processed_pairs)
split = int(len(processed_pairs) * 0.9)
train_pairs = processed_pairs[:split]
test_pairs  = processed_pairs[split:]

# ── Build vocabularies on training data only ─────────────────────
src_vocab = Vocabulary('English')
tgt_vocab = Vocabulary('Hindi')

for en, hi in train_pairs:
    src_vocab.count(en)
    tgt_vocab.count(hi)

min_freq  = 1 if not USING_REAL_DATA else 2
src_vocab.build(min_freq=min_freq, max_vocab=10000)
tgt_vocab.build(min_freq=min_freq, max_vocab=15000)

print(f'Source (English) vocab : {len(src_vocab):,} words')
print(f'Target (Hindi)   vocab : {len(tgt_vocab):,} words')
print(f'Training pairs         : {len(train_pairs):,}')
print(f'Test pairs             : {len(test_pairs):,}')

# OOV rate
sample = train_pairs[:min(1000, len(train_pairs))]
def oov_pct(pairs, vocab, lang_idx):
    total = unk = 0
    for p in pairs:
        for w in p[lang_idx].split():
            total += 1
            if w not in vocab.w2i:
                unk += 1
    return 100 * unk / total if total else 0

print(f'OOV rate (train sample) — EN: {oov_pct(sample, src_vocab, 0):.2f}%  '
      f'HI: {oov_pct(sample, tgt_vocab, 1):.2f}%')

In [ ]:
# ── Dataset & DataLoader with padding ────────────────────────────

class TranslationDataset(Dataset):
    """
    Converts sentence pairs to integer tensors.
    Source: word tokens → integer IDs (with EOS)
    Target: word tokens → integer IDs (with SOS + EOS, for teacher forcing)
    """
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.data = pairs
        self.sv   = src_vocab
        self.tv   = tgt_vocab

    def __len__(self):  return len(self.data)

    def __getitem__(self, i):
        en, hi = self.data[i]
        src = torch.tensor(self.sv.encode(en, add_sos=False, add_eos=True),  dtype=torch.long)
        tgt = torch.tensor(self.tv.encode(hi, add_sos=True,  add_eos=True),  dtype=torch.long)
        return src, tgt


def collate_fn(batch):
    """
    Pad variable-length sequences in a batch to the same length.
    Padding is done with PAD_IDX = 0.
    Returns:
        src_padded : [batch_size, max_src_len]
        tgt_padded : [batch_size, max_tgt_len]
    """
    src_seqs, tgt_seqs = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_seqs, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_seqs, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


BATCH_SIZE = 64

train_ds  = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
test_ds   = TranslationDataset(test_pairs,  src_vocab, tgt_vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

# Verify shapes
src_b, tgt_b = next(iter(train_loader))
print(f'✓ Train DataLoader : {len(train_loader)} batches × {BATCH_SIZE}')
print(f'✓ Test  DataLoader : {len(test_loader)} batches × {BATCH_SIZE}')
print(f'  Sample batch — src: {list(src_b.shape)}  tgt: {list(tgt_b.shape)}')
print(f'  (shape = [batch_size, padded_seq_len])')

---
## Task 2: Encoder–Decoder Model Design (3 Marks)

### Architecture Diagram

```
English input tokens
        │
        ▼
┌─────────────────────────────────┐
│           ENCODER               │
│                                 │
│  ①  Embedding Layer            │
│      token_id → dense vector    │
│      shape: [B, T, embed_dim]   │
│                                 │
│  ②  GRU (stacked, 2 layers)    │
│      reads full source sequence │
│      captures order & context   │
│                                 │
│  ③  Final Hidden State h        │
│      [num_layers, B, hidden]    │
│      = compressed meaning       │
└──────────────┬──────────────────┘
               │  h  (context vector)
               ▼
┌─────────────────────────────────┐
│           DECODER               │
│  (step-by-step token generation)│
│                                 │
│  ④  Embedding Layer            │
│      prev token → dense vector  │
│                                 │
│  ⑤  GRU (init from encoder h)  │
│      generates hidden state     │
│      at each decoding step      │
│                                 │
│  ⑥  Linear + Softmax           │
│      hidden → vocab logits      │
│      pick argmax → next token   │
└─────────────────────────────────┘
        │
        ▼
 Hindi output tokens
```

### Role of Each Layer

| # | Layer | Where | Role |
|---|-------|-------|------|
| ① | Embedding | Encoder | Maps discrete token IDs to continuous dense vectors. Lets similar words be close in vector space. `padding_idx=0` keeps `<PAD>` at zero. |
| ② | GRU | Encoder | Reads source tokens left-to-right, maintaining a running hidden state that accumulates context. GRU gates (reset/update) control information flow without the vanishing gradient of plain RNNs. |
| ③ | Hidden State | Encoder→Decoder | The final encoder hidden state is the "thought vector" — a fixed-size summary of the entire source sentence — passed as the decoder's initial state. |
| ④ | Embedding | Decoder | Converts the previously generated (or during teacher forcing: reference) token into a vector for the decoder GRU. |
| ⑤ | GRU | Decoder | Generates a new hidden state at each step conditioned on the previous token embedding + previous hidden state. This hidden state carries the partially-generated translation history. |
| ⑥ | Linear + Softmax | Decoder | Projects the GRU hidden state to a logit vector of size `tgt_vocab`, then softmax to get probabilities. The highest-probability token is selected as output. |

In [ ]:
class Encoder(nn.Module):
    """
    GRU Encoder.

    Input  → Embedding → GRU → (all_outputs, final_hidden)

    The final_hidden state encodes the entire source sentence and is
    passed to the Decoder as its initial hidden state.
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor):
        # src: [B, src_len]
        emb = self.dropout(self.embedding(src))          # [B, src_len, E]
        outputs, hidden = self.gru(emb)                  # [B, src_len, H], [L, B, H]
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    """
    GRU Decoder (single-step).

    At each decoding step, receives:
      - tgt_token : the previous output token  [B]
      - hidden    : previous hidden state       [L, B, H]

    Returns:
      - logits : raw scores over target vocab   [B, tgt_vocab]
      - hidden : updated hidden state           [L, B, H]
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=PAD_IDX)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc_out  = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt_token: torch.Tensor,
                hidden: torch.Tensor):
        # tgt_token: [B]  →  unsqueeze to [B, 1]
        emb = self.dropout(self.embedding(tgt_token.unsqueeze(1)))  # [B, 1, E]
        out, hidden = self.gru(emb, hidden)                         # [B, 1, H], [L, B, H]
        logits = self.fc_out(out.squeeze(1))                        # [B, tgt_vocab]
        return logits, hidden

In [ ]:
class Seq2Seq(nn.Module):
    """
    Full Encoder-Decoder Seq2Seq model.

    Teacher Forcing:
      During training, with probability `tf_ratio` the decoder is fed
      the *ground-truth* previous token instead of its own prediction.
      This stabilises training early on and leads to faster convergence.
      At inference (tf_ratio=0) the decoder always uses its own output.
    """
    def __init__(self, encoder: Encoder, decoder: Decoder,
                 device: torch.device):
        super().__init__()
        assert encoder.gru.hidden_size == decoder.gru.hidden_size
        assert encoder.gru.num_layers  == decoder.gru.num_layers
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                tf_ratio: float = 0.5) -> torch.Tensor:
        """
        Args:
            src      : [B, src_len]  — source token IDs
            tgt      : [B, tgt_len]  — target token IDs (SOS ... EOS)
            tf_ratio : teacher-forcing probability
        Returns:
            outputs  : [B, tgt_len-1, tgt_vocab]  — logits at each step
        """
        B, T      = src.shape[0], tgt.shape[1]
        V         = self.decoder.fc_out.out_features
        outputs   = torch.zeros(B, T - 1, V, device=self.device)

        _, hidden = self.encoder(src)       # encode source
        dec_in    = tgt[:, 0]               # first input = <SOS>

        for t in range(1, T):
            logits, hidden = self.decoder(dec_in, hidden)
            outputs[:, t - 1] = logits
            # teacher forcing: use ground-truth or model prediction
            dec_in = tgt[:, t] if random.random() < tf_ratio else logits.argmax(1)

        return outputs   # [B, T-1, V]

    @torch.no_grad()
    def translate(self, src: torch.Tensor, max_len: int = 60) -> List[int]:
        """Greedy translation for a single (or batched) source sentence."""
        self.eval()
        _, hidden = self.encoder(src)
        dec_in    = torch.tensor([SOS_IDX], device=self.device)
        predicted = []
        for _ in range(max_len):
            logits, hidden = self.decoder(dec_in, hidden)
            top1 = logits.argmax(1)
            if top1.item() == EOS_IDX:
                break
            predicted.append(top1.item())
            dec_in = top1
        return predicted


# ── Hyperparameters ──────────────────────────────────────────────
EMBED_DIM  = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT    = 0.3
LR         = 1e-3
N_EPOCHS   = 10
TF_RATIO   = 0.5    # teacher-forcing ratio
CLIP       = 1.0    # gradient clipping

# ── Model ────────────────────────────────────────────────────────
enc   = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
dec   = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

# Xavier uniform initialisation
def init_weights(m):
    for n, p in m.named_parameters():
        nn.init.xavier_uniform_(p.data) if 'weight' in n else nn.init.constant_(p.data, 0)
model.apply(init_weights)

# Parameter count
def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print('=' * 52)
print(f'{"Model Summary":^52}')
print('=' * 52)
print(f'  Encoder params : {n_params(enc):>12,}')
print(f'  Decoder params : {n_params(dec):>12,}')
print(f'  Total params   : {n_params(model):>12,}')
print(f'  Embed dim      : {EMBED_DIM}')
print(f'  Hidden dim     : {HIDDEN_DIM}')
print(f'  GRU layers     : {NUM_LAYERS}')
print(f'  Dropout        : {DROPOUT}')
print('=' * 52)
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Learning rate  : {LR}')
print(f'  Epochs         : {N_EPOCHS}')
print(f'  Teacher forcing: {TF_RATIO:.0%}')
print('=' * 52)

---
## Task 3: Model Training (3 Marks)

### Training Setup

| Component | Choice | Reason |
|-----------|--------|--------|
| **Loss** | Cross-Entropy (`ignore_index=<PAD>`) | Padding positions do not contribute to loss |
| **Optimizer** | Adam (lr=1e-3, wd=1e-5) | Adaptive learning rates; fast convergence for NMT |
| **Teacher Forcing** | 50% ratio | Balances training stability vs exposure bias |
| **Gradient Clip** | max-norm = 1.0 | Prevents exploding gradients in RNNs |
| **LR Scheduler** | ReduceLROnPlateau (×0.5 after 2 bad epochs) | Reduces LR when val loss plateaus |

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)


def train_one_epoch(model, loader, optimizer, criterion,
                    clip, tf_ratio):
    """Single training epoch. Returns mean cross-entropy loss."""
    model.train()
    epoch_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        optimizer.zero_grad()

        output = model(src, tgt, tf_ratio)          # [B, T-1, V]
        V = output.shape[-1]
        loss = criterion(output.reshape(-1, V),     # [(B*(T-1)), V]
                         tgt[:, 1:].reshape(-1))    # [(B*(T-1))]  skip <SOS>
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    """Evaluation pass (no teacher forcing, no gradient). Returns mean loss."""
    model.eval()
    epoch_loss = 0.0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            output   = model(src, tgt, tf_ratio=0.0)
            V        = output.shape[-1]
            loss     = criterion(output.reshape(-1, V), tgt[:, 1:].reshape(-1))
            epoch_loss += loss.item()
    return epoch_loss / len(loader)


print('✓ Training functions defined')

In [ ]:
train_losses, val_losses = [], []
best_val = float('inf')
BEST_MODEL = 'best_nmt_model.pt'

print(f'{"-"*62}')
print(f'{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>9} | '
      f'{"Train PPL":>10} | {"Val PPL":>8} | {"Time":>6}')
print(f'{"-"*62}')

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()

    tr_loss = train_one_epoch(model, train_loader, optimizer,
                               criterion, CLIP, TF_RATIO)
    vl_loss = evaluate(model, test_loader, criterion)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    scheduler.step(vl_loss)

    mark = ''
    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(model.state_dict(), BEST_MODEL)
        mark = ' ←'

    elapsed = time.time() - t0
    print(f'{epoch:>6} | {tr_loss:>11.4f} | {vl_loss:>9.4f} | '
          f'{np.exp(tr_loss):>10.2f} | {np.exp(vl_loss):>8.2f} | {elapsed:>5.1f}s{mark}')

print(f'{"-"*62}')
print(f'Best Val Loss : {best_val:.4f}  (PPL {np.exp(best_val):.2f})')

# Load best model
model.load_state_dict(torch.load(BEST_MODEL, map_location=DEVICE))
print('✓ Best model loaded')

In [ ]:
epochs_x = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_x, train_losses, 'b-o', lw=2, ms=5, label='Train Loss')
axes[0].plot(epochs_x, val_losses,   'r-o', lw=2, ms=5, label='Val Loss')
axes[0].set_title('Cross-Entropy Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xticks(epochs_x)

# Perplexity
tr_ppl = [np.exp(l) for l in train_losses]
vl_ppl = [np.exp(l) for l in val_losses]
axes[1].plot(epochs_x, tr_ppl, 'b-o', lw=2, ms=5, label='Train PPL')
axes[1].plot(epochs_x, vl_ppl, 'r-o', lw=2, ms=5, label='Val PPL')
axes[1].set_title('Perplexity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xticks(epochs_x)

plt.suptitle('Training History — GRU Encoder-Decoder (en→hi)', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>9} | {"Train PPL":>10} | {"Val PPL":>8}')
print('─' * 55)
for e, (tl, vl) in enumerate(zip(train_losses, val_losses), 1):
    marker = ' ←' if vl == min(val_losses) else ''
    print(f'{e:>6} | {tl:>11.4f} | {vl:>9.4f} | {np.exp(tl):>10.2f} | {np.exp(vl):>8.2f}{marker}')

---
## Task 4: Translation Evaluation (4 Marks)

### Evaluation Protocol

- **25 unseen test sentences** translated with greedy decoding
- **BLEU Score** (corpus + sentence-level) using SacreBLEU
- **Side-by-side** reference vs prediction comparison
- **Error analysis** across four categories

In [ ]:
def translate(sentence: str, model: Seq2Seq,
              src_vocab: Vocabulary, tgt_vocab: Vocabulary,
              max_len: int = 60) -> str:
    """
    Translate a single English sentence to Hindi using greedy decoding.

    Steps:
      1. Preprocess & tokenize the English input
      2. Encode to integer IDs using src_vocab
      3. Run encoder → get hidden state
      4. Decoder loop: at each step pick argmax token until <EOS> or max_len
      5. Decode predicted IDs back to Hindi words
    """
    model.eval()
    processed  = preprocess_english(sentence)
    src_ids    = src_vocab.encode(processed, add_sos=False, add_eos=True)
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)

    pred_ids   = model.translate(src_tensor, max_len=max_len)
    return tgt_vocab.decode(pred_ids)


print('✓ translate() ready')

In [ ]:
N_EVAL = 25
eval_pairs = random.sample(test_pairs, min(N_EVAL, len(test_pairs)))

results = []
print(f'Translating {N_EVAL} unseen sentences …\n')
print('─' * 80)

for i, (en_ref, hi_ref) in enumerate(eval_pairs, 1):
    hi_pred = translate(en_ref, model, src_vocab, tgt_vocab)
    results.append(dict(idx=i, source=en_ref, reference=hi_ref, prediction=hi_pred))

    print(f'[{i:2d}] EN (source)    : {en_ref}')
    print(f'     HI (reference) : {hi_ref}')
    print(f'     HI (predicted) : {hi_pred if hi_pred else "<empty>"}')
    print()

print(f'✓ Translated {len(results)} sentences')

In [ ]:
# ── BLEU computation ─────────────────────────────────────────────
bleu_metric = BLEU(effective_order=True)

hypotheses = [r['prediction'] for r in results]
references = [[r['reference'] for r in results]]    # list of lists

# Corpus-level BLEU
corpus_score = bleu_metric.corpus_score(hypotheses, references)

# Sentence-level BLEU
sent_scores = [
    bleu_metric.sentence_score(h, [r]).score
    for h, r in zip(hypotheses, references[0])
]

print('=' * 55)
print(f'{"BLEU Evaluation Results":^55}')
print('=' * 55)
print(f'  Corpus BLEU : {corpus_score}')
print(f'  Sent BLEU — Mean   : {np.mean(sent_scores):.2f}')
print(f'              Median : {np.median(sent_scores):.2f}')
print(f'              Max    : {np.max(sent_scores):.2f}')
print(f'              Min    : {np.min(sent_scores):.2f}')
print('=' * 55)

# Accuracy (% sentences with BLEU ≥ 20)
acc_threshold = 20.0
accuracy = sum(1 for s in sent_scores if s >= acc_threshold) / len(sent_scores) * 100
print(f'  Accuracy (BLEU ≥ {acc_threshold}) : {accuracy:.1f}%')

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(sent_scores, bins=15, color='mediumseagreen', alpha=0.85, edgecolor='white')
axes[0].axvline(np.mean(sent_scores), color='red', ls='--',
                label=f'Mean = {np.mean(sent_scores):.1f}')
axes[0].set_xlabel('Sentence BLEU'); axes[0].set_ylabel('Count')
axes[0].set_title('BLEU Score Distribution'); axes[0].legend(); axes[0].grid(alpha=0.3)

ranked = sorted(enumerate(sent_scores, 1), key=lambda x: x[1], reverse=True)
axes[1].bar([r[0] for r in ranked], [r[1] for r in ranked],
            color='steelblue', alpha=0.85)
axes[1].set_xlabel('Sentence (ranked)'); axes[1].set_ylabel('BLEU')
axes[1].set_title('Per-Sentence BLEU (Ranked)'); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Translation BLEU Evaluation', fontsize=13)
plt.tight_layout()
plt.savefig('bleu_scores.png', dpi=150, bbox_inches='tight')
plt.show()

# Ranked table
print(f'\n{"#":>3} | {"BLEU":>6} | Source (EN)')
print('─' * 70)
for r, s in sorted(zip(results, sent_scores), key=lambda x: x[1], reverse=True):
    print(f'{r["idx"]:>3} | {s:>6.1f} | {r["source"][:60]}')

In [ ]:
# ── Error Analysis ────────────────────────────────────────────────
def categorise_error(r, score):
    ref_words  = set(r['reference'].split())
    pred_words = r['prediction'].split()
    ref_len    = len(r['reference'].split())
    pred_len   = len(pred_words)

    if UNK in r['prediction']:
        return 'Unknown Words'
    if pred_len == 0 or (ref_len > 0 and pred_len / ref_len < 0.4):
        return 'Missing Words'
    overlap = len(set(pred_words) & ref_words) / max(len(ref_words), 1)
    if overlap < 0.25:
        return 'Word Order / Vocabulary'
    if pred_len > 0 and ref_len > 0 and abs(pred_len - ref_len) / ref_len > 0.5:
        return 'Incorrect Grammar'
    return 'Good'


categories = [categorise_error(r, s) for r, s in zip(results, sent_scores)]
cat_counts  = Counter(categories)

print('Translation Error Analysis')
print('=' * 65)
for cat, count in cat_counts.most_common():
    example = next((r for r, c in zip(results, categories) if c == cat), None)
    marker  = '✓' if cat == 'Good' else '✗'
    print(f'\n{marker} {cat} ({count} / {len(results)} sentences)')
    if example:
        print(f'  Example — EN  : {example["source"]}')
        print(f'             REF: {example["reference"]}')
        print(f'             HYP: {example["prediction"] or "<empty>"}')

# Pie chart
labels = list(cat_counts.keys())
sizes  = [cat_counts[l] for l in labels]
colors = ['#2ecc71','#e74c3c','#f39c12','#3498db','#9b59b6'][:len(labels)]

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, pcts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=140, pctdistance=0.82
)
for t in texts + pcts:
    t.set_fontsize(10)
ax.set_title('Translation Error Distribution', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('error_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 5: Analysis and Conclusion (1 Mark)

### Q1. Advantages of Encoder-Decoder Models over Statistical Machine Translation (SMT)

| Aspect | SMT (Phrase-Based) | Encoder-Decoder (Neural) |
|--------|-------------------|---------------------------|
| **Representation** | Sparse phrase tables, hand-crafted features | Dense continuous embeddings learned end-to-end |
| **Context** | Limited to local n-gram windows | GRU captures long-range sequential dependencies |
| **Fluency** | Needs a separate language model | Translation + fluency are jointly optimised |
| **Morphology** | Poor on Hindi (rich morphology, agglutination) | Embeddings generalise over morphological variants |
| **Word alignment** | Required as a separate step | Implicit in encoder hidden states |
| **Training** | Multi-stage pipeline (alignment → phrase table → LM → decoder) | Single end-to-end objective |
| **Generalisation** | Fails on unseen phrase combinations | Compositionality via continuous space |
| **Domain adaptation** | Requires re-building phrase tables | Fine-tune with small in-domain corpus |

**Key point:** NMT learns *what to translate* and *how to say it* jointly in one model, making it significantly better on morphologically-rich, SOV languages like Hindi where phrase reordering is frequent.

---

### Q2. Limitations Observed in This Translation System

1. **Fixed-size bottleneck** — The entire source sentence is compressed into a single hidden state vector. For sentences longer than ~15 tokens, information is lost before decoding begins. This is the primary limitation of vanilla Seq2Seq without attention.

2. **OOV / Unknown words** — Hindi has rich morphology; many inflected forms fall outside the training vocabulary and map to `<UNK>`, breaking the output.

3. **Word-order errors** — English uses SVO word order; Hindi uses SOV. The model often generates translations with English-style ordering, producing grammatically incorrect Hindi.

4. **Missing words in long sentences** — For sentences > 20 tokens, the decoder tends to terminate early, losing information at the tail of the source.

5. **Gender/number agreement** — Hindi verbs and adjectives must agree with the subject in gender and number. The model produces inconsistent agreement, particularly for unseen subject-verb pairs.

6. **Exposure bias** — Teacher forcing during training means the model never recovers from its own errors; at inference, early mistakes cascade.

---

### Q3. Proposed Improvement: Bahdanau Attention Mechanism

**Problem it solves:** The fixed-size hidden-state bottleneck.

**Mechanism:** Instead of relying on only the final encoder hidden state, attention lets the decoder *attend to every encoder hidden state* at each decoding step:

$$e_{t,s} = v^\top \tanh\bigl(W_a\, h_t^{\text{dec}} + U_a\, h_s^{\text{enc}}\bigr)$$

$$\alpha_{t,s} = \text{softmax}(e_{t,s})\qquad c_t = \sum_s \alpha_{t,s}\, h_s^{\text{enc}}$$

The context vector $c_t$ is concatenated with the decoder hidden state before the output projection:

$$P(y_t \mid y_{<t}, x) = \text{softmax}\bigl(W_o\,[h_t^{\text{dec}};\, c_t]\bigr)$$

**Expected improvements:**
- Eliminates the information bottleneck → better BLEU on long sentences
- Implicit word alignment (attention weights show which source words drive each target token)
- Reduced SOV reordering errors (decoder can re-attend to earlier source positions)
- Interpretable alignment matrix for error analysis

**Alternative — Transformer Architecture:** Replace the entire RNN-based encoder-decoder with the Transformer (Vaswani et al., 2017). Multi-head self-attention + positional encodings eliminate sequential processing, enabling:
- Fully parallel training (10–100× faster than GRU)
- Better global dependency modelling
- Access to pre-trained models: **IndicTrans2** (AI4Bharat, 2023) achieves state-of-the-art en↔hi translation with BLEU > 40 by pretraining on 100M+ sentence pairs.

---
## 📸 Screenshots — BITS CSIS Lab Execution (Instruction 6)

> **Instructions for team:** Run this notebook on **BITS OSHA Cloud Lab / CSIS Lab**, then paste screenshots below showing:
> 1. Lab login / environment (proving BITS CSIS Lab was used)
> 2. Each task cell executing without errors
> 3. Training loss output
> 4. BLEU score output
> 5. Sample translations output

---

### Screenshot 1 — BITS CSIS Lab Login / Environment

*[ Paste screenshot here ]*

---

### Screenshot 2 — Task 1: Dataset & Preprocessing Output

*[ Paste screenshot here ]*

---

### Screenshot 3 — Task 2: Model Summary Output

*[ Paste screenshot here ]*

---

### Screenshot 4 — Task 3: Training Loop (loss per epoch)

*[ Paste screenshot here ]*

---

### Screenshot 5 — Task 3: Loss Curves Plot

*[ Paste screenshot here ]*

---

### Screenshot 6 — Task 4: Sample Translations (25 sentences)

*[ Paste screenshot here ]*

---

### Screenshot 7 — Task 4: BLEU Score Output

*[ Paste screenshot here ]*

---

### Screenshot 8 — Task 4: Error Analysis Plot

*[ Paste screenshot here ]*

In [ ]:
# ── Final Summary ────────────────────────────────────────────────
print('=' * 65)
print(f'{"NLP Assignment 2 — Final Summary":^65}')
print('=' * 65)
print(f'  Language pair          : English → Hindi')
print(f'  Dataset                : Samanantar (en-hi)')
print(f'  Architecture           : GRU Encoder-Decoder + Teacher Forcing')
print(f'  Source vocabulary      : {len(src_vocab):,} words')
print(f'  Target vocabulary      : {len(tgt_vocab):,} words')
print(f'  Model parameters       : {n_params(model):,}')
print(f'  Training pairs         : {len(train_pairs):,}')
print(f'  Test pairs             : {len(test_pairs):,}')
print(f'  Epochs trained         : {N_EPOCHS}')
print(f'  Batch size             : {BATCH_SIZE}')
print(f'  Learning rate          : {LR}')
print(f'  Teacher forcing ratio  : {TF_RATIO:.0%}')
print(f'  Best val loss          : {best_val:.4f}  (PPL {np.exp(best_val):.2f})')
print(f'  Corpus BLEU (25 sents) : {corpus_score}')
print(f'  Mean sentence BLEU     : {np.mean(sent_scores):.2f}')
print('=' * 65)
print('\n✓ All 5 tasks complete.')